# SaRoHead - zero-/few-shot RoLLMs

Needs a 24 GB+ GPU (A100/L4). Zero-shot is deterministic and run once.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/sarohead")
REPO = Path("/content/SaRoHead")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/GRAI-UNSTPB/SaRoHead.git", str(REPO)], check=True)

RESULTS = DRIVE / "results" / "predictions"
RESULTS.mkdir(parents=True, exist_ok=True)
os.environ["SAROHEAD_RESULTS"] = str(RESULTS)
sys.path.insert(0, str(REPO))
from analysis_scripts.prediction_io import prediction_filename

SEEDS = [13, 42, 123, 2024, 7]
CATEGORIES = ["social", "politic", "sport"]


def run(cmd, cwd):
    print(">>", " ".join(map(str, cmd)), flush=True)
    subprocess.run([str(c) for c in cmd], cwd=str(cwd), check=True)


def done(method, model, variant, category, seed):
    return (RESULTS / method / prediction_filename(method, model, variant, category, seed, "random", "test")).exists()


In [ ]:
!pip install -q vllm transformers accelerate
!pip uninstall -y -q torchaudio

MODELS = ["llama2", "llama3", "gemma", "mistral"]
N_EXAMPLES = [0, 2, 4, 6]
PAPER = {"llama2": "RoLlama2", "llama3": "RoLlama3", "gemma": "RoGemma", "mistral": "RoMistral"}


In [ ]:
for seed in SEEDS:
    for model in MODELS:
        for n in N_EXAMPLES:
            if n == 0 and seed != SEEDS[0]:
                continue
            if all(done("few_shot", PAPER[model], f"n{n}", c, seed) for c in CATEGORIES):
                continue
            run([sys.executable, "few_shot_LLM_classif.py", "--model_type", model, "--n_examples", n, "--seed", seed],
                REPO / "LLM_scripts")


In [ ]:
files = sorted(RESULTS.rglob("*.csv"))
print(len(files), "prediction files in", RESULTS)
